# 12g — Dipole Angle vs MJD for a selected diaObject

## Purpose

Plot the **dipole orientation angle** (`r:dipoleAngle`) as a function of observation time (MJD TAI)
for a user-selected `diaObjectId`, in order to investigate whether the dipole direction is
random, clustered, or drifting over time — which would be a key discriminant between
PSF-mismatch artefacts and genuine astrophysical variability.

## Angular representation — which convention?

The Rubin AP pipeline stores `dipoleAngle` as the **position angle of the positive lobe**
of the dipole, measured East of North on the sky, in degrees.  Because a dipole is a
two-signed object (positive lobe ↔ negative lobe), the *physical* orientation has a
**period of 180°**, not 360°.  Three representations are therefore provided:

| Convention | Range | Notes |
|---|---|---|
| `[0°, 360°)` | standard, always positive | Easy to read but 0°/360° wrapping artefacts |
| `(−180°, +180°]` | signed, North=0 | Most natural for scatter around a mean |
| `[0°, 180°)` **mod 180°** | orientation (undirected) | Correct physical period for a dipole |

> **Recommendation:** for a dipole the **mod-180° representation** is the physically
> correct one. Use `[−90°, +90°]` (i.e. shift mod-180° by −90°) if you want to
> centre the distribution around zero for comparison across visits.

## Figure configurations produced

1. **`fig_stacked`** — `(N_band × 1)` stacked subplots, one per band, **three angle conventions** shown
   as separate curves (or a drop-down selection via `ANGLE_CONV` parameter).
2. **`fig_overlay`** — single subplot with **all bands overlaid** (one colour per band).

Both figures are produced for each of the three angle conventions.

## Data source

Same `fullcutouts_{diaObjectId}/manifest.csv` as notebook 12b — no additional download needed.

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-16
- **Subject:** Fink/LSST DIA — Dipole angle temporal analysis


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────

# diaObjectId to inspect — must have a fullcutouts_{id}/ directory
objsid = {
    0: 170028527433285667,
    1: 170028526498480187,
    2: 170028485669027856,
    3: 313893022845108234,
    4: 170028526632697879,
    5: 170028529316003913,
}

DIAOBJECT_ID = objsid[1]

# Bands to display: None = all available, or e.g. ["r", "i"]
BANDS_FILTER = None

# Angle convention to highlight in the stacked/overlay figures.
# Options: '0_360'  |  '-180_180'  |  'mod180'
# The notebook always produces ALL three conventions but this selects
# which one goes into the dedicated overlay figure.
ANGLE_CONV_PRIMARY = "-180_180"

# Only plot visits flagged as isDipole=True?
# False → plot all visits that have a finite dipoleAngle value
DIPOLE_ONLY = False

# Marker size and alpha
MARKER_SIZE = 10
MARKER_ALPHA = 0.85

# Save figures?
SAVE_FIGS = True
DIR_FIGS = "figs_FINK_BLOCK_LC_12g"

# ─────────────────────────────────────────────────────────────────────────────
# Fixed paths (same as notebook 12b)
# ─────────────────────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest.csv")
FILE_GAIA_MATCHED = os.path.join("data_FINK_BLOCK_LC_08b", "fink_diaobj_gaia_join_matched.csv")

BAND_ORDER = list("ugrizy")
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}

os.makedirs(DIR_FIGS, exist_ok=True)
print(f"diaObjectId : {DIAOBJECT_ID}")
print(f"Manifest    : {os.path.abspath(FILE_MANIFEST)}")
print(f"Figures     : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def mjd_to_date(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


def parse_bool(val):
    """Coerce a dipole flag (bool / int / str / NaN) to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


# ── Angular convention converters ────────────────────────────────────────────


def angle_0_360(deg):
    """Map angle array to [0°, 360°)."""
    return np.mod(np.asarray(deg, dtype=float), 360.0)


def angle_m180_p180(deg):
    """Map angle array to (−180°, +180°]."""
    a = np.mod(np.asarray(deg, dtype=float), 360.0)
    return np.where(a > 180.0, a - 360.0, a)


def angle_mod180(deg):
    """Map angle array to [0°, 180°) — undirected dipole orientation."""
    return np.mod(np.asarray(deg, dtype=float), 180.0)


def apply_convention(deg, conv):
    """Dispatch to the requested angular convention."""
    if conv == "0_360":
        return angle_0_360(deg)
    elif conv == "-180_180":
        return angle_m180_p180(deg)
    elif conv == "mod180":
        return angle_mod180(deg)
    else:
        raise ValueError(f"Unknown convention: {conv!r}  (use '0_360', '-180_180', or 'mod180')")


# Convention metadata: label, y-axis limits, reference lines
CONV_META = {
    "0_360": {
        "label": "Dipole angle [°]  — [0°, 360°)",
        "ylim": (-5, 365),
        "yticks": [0, 90, 180, 270, 360],
        "hlines": [0, 90, 180, 270, 360],
    },
    "-180_180": {
        "label": "Dipole angle [°]  — (−180°, +180°]",
        "ylim": (-185, 185),
        "yticks": [-180, -90, 0, 90, 180],
        "hlines": [-180, -90, 0, 90, 180],
    },
    "mod180": {
        "label": "Dipole angle [°]  — mod 180° (undirected orientation)",
        "ylim": (-5, 185),
        "yticks": [0, 45, 90, 135, 180],
        "hlines": [0, 90, 180],
    },
}

print("Utility functions defined.")

## 3. Load manifest

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(
        f"{FILE_MANIFEST} not found.\nRun notebook 12a / fink_download_full_cutouts.py for obj {DIAOBJECT_ID}"
    )

df = pd.read_csv(FILE_MANIFEST)
df["isDipole"] = df["r:isDipole"].fillna(False).apply(parse_bool)
df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)

# Keep only rows with a finite dipoleAngle
has_angle = df["r:dipoleAngle"].notna() & np.isfinite(df["r:dipoleAngle"].astype(float))
if DIPOLE_ONLY:
    df_ang = df[df["isDipole"] & has_angle].copy()
    print(f"Keeping isDipole=True rows only: {len(df_ang)} / {len(df)}")
else:
    df_ang = df[has_angle].copy()
    print(f"All rows with finite dipoleAngle: {len(df_ang)} / {len(df)}")

df_ang["r:dipoleAngle"] = df_ang["r:dipoleAngle"].astype(float)

n_dip = df["isDipole"].sum()
print(f"Total diaSources : {len(df)}   |   isDipole=True : {n_dip}  ({100 * n_dip / len(df):.1f}%)")
print(f"Bands available  : {sorted(df_ang['r:band'].unique())}")
df_ang[
    [
        "r:diaSourceId",
        "r:band",
        "r:midpointMjdTai",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleFluxDiff",
        "isDipole",
    ]
].head(10)

## 4. Load Gaia DR3 object-level metadata

In [ ]:
gaia_G_mag, gaia_status, fink_group, gaia_dr3_id = np.nan, "?", "?", "?"

if os.path.exists(FILE_GAIA_MATCHED):
    df_gaia = pd.read_csv(FILE_GAIA_MATCHED)
    hit = df_gaia[df_gaia["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        gaia_G_mag = float(r.get("gaia_phot_g_mean_mag", np.nan))
        gaia_status = str(r.get("gaia_status", "?"))
        fink_group = str(r.get("group", "?"))
        gaia_dr3_id = str(r.get("dr3_source_id", "?"))

G_str = f"{gaia_G_mag:.2f}" if np.isfinite(gaia_G_mag) else "?"
print(f"Gaia G={G_str}  |  status={gaia_status}  |  group={fink_group}  |  DR3={gaia_dr3_id}")

## 5. Temporal reference

In [ ]:
T0_MJD = df["r:midpointMjdTai"].min()
T0_DATE = mjd_to_date(T0_MJD)
DATE_FIRST = mjd_to_date(df["r:midpointMjdTai"].min())
DATE_LAST = mjd_to_date(df["r:midpointMjdTai"].max())
TSPAN_DAYS = df["r:midpointMjdTai"].max() - T0_MJD

print(f"Time origin  t0 = MJD {T0_MJD:.4f}  ({T0_DATE})")
print(f"First detection : {DATE_FIRST}   Last : {DATE_LAST}   Span : {TSPAN_DAYS:.1f} d")

# Add delta-t column to the angle dataframe
df_ang["dt_days"] = df_ang["r:midpointMjdTai"].values.astype(float) - T0_MJD

## 6. Helper: per-band angle statistics

In [ ]:
def circular_mean_std(angles_deg):
    """Compute circular mean and circular std (degrees) for an array of angles.

    Uses the standard complex-exponential estimator:
      R = |mean(exp(i*theta))|
      mean_angle = arg(mean(exp(i*theta)))
      circular_std = sqrt(-2 * ln(R))  [radians → degrees]

    Parameters
    ----------
    angles_deg : array_like
        Angles in degrees.

    Returns
    -------
    mu_deg  : float — circular mean in degrees
    std_deg : float — circular std in degrees
    R       : float — mean resultant length in [0, 1] (1 = perfectly clustered)
    """
    theta = np.deg2rad(np.asarray(angles_deg, dtype=float))
    z = np.exp(1j * theta)
    zbar = np.nanmean(z)
    R = float(np.abs(zbar))
    mu = float(np.angle(zbar, deg=True))  # degrees, in (−180, +180]
    std = float(np.rad2deg(np.sqrt(max(-2.0 * np.log(R + 1e-15), 0.0))))
    return mu, std, R


def band_angle_stats(df_b, conv):
    """Return a dict of stats for band subset df_b, angle convention conv."""
    ang = apply_convention(df_b["r:dipoleAngle"].values, conv)
    mu_c, std_c, R_c = circular_mean_std(ang)
    return {
        "N": len(ang),
        "N_dip": int(df_b["isDipole"].sum()),
        "mean": float(np.nanmean(ang)),
        "median": float(np.nanmedian(ang)),
        "std": float(np.nanstd(ang)),
        "circ_mean": mu_c,
        "circ_std": std_c,
        "R": R_c,
    }


print("Statistics helpers defined.")

## 7. Configuration: bands to plot

In [ ]:
bands_available = [b for b in BAND_ORDER if b in df_ang["r:band"].values]
bands_to_plot = [b for b in bands_available if BANDS_FILTER is None or b in BANDS_FILTER]
N_bands = len(bands_to_plot)

print(f"Bands with dipole angle data : {bands_available}")
print(f"Bands to plot                : {bands_to_plot}")

# Quick summary table
print("\nAngle statistics per band (primary convention: " + ANGLE_CONV_PRIMARY + "):")
for b in bands_to_plot:
    db = df_ang[df_ang["r:band"] == b]
    st = band_angle_stats(db, ANGLE_CONV_PRIMARY)
    print(
        f"  band {b}  N={st['N']:3d}  N_dip={st['N_dip']:3d}  "
        f"circ_mean={st['circ_mean']:+7.2f}°  circ_std={st['circ_std']:6.2f}°  R={st['R']:.3f}"
    )

## 8. Figure A — Stacked subplots (N_band × 1), one convention at a time

Three figures produced, one per angular convention (`0_360`, `-180_180`, `mod180`).
Each subplot shows the dipole angle vs MJD for one band.
Dipole-flagged visits are highlighted with a gold ring.
Reference guide-lines are drawn at cardinal angles.
A stats box shows N, circular mean ± circular std, and R.

In [ ]:
def _stats_box(ax, stats, conv):
    """Overlay a compact stats annotation on the angle subplot."""
    txt = (
        f"N={stats['N']}  N_dip={stats['N_dip']}\n"
        f"circ mean = {stats['circ_mean']:+.1f}°\n"
        f"circ std  =  {stats['circ_std']:.1f}°\n"
        f"R = {stats['R']:.3f}"
    )
    ax.text(
        0.01,
        0.97,
        txt,
        transform=ax.transAxes,
        fontsize=7,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#aaa", alpha=0.85),
    )


def plot_stacked(conv):
    """Produce the (N_band × 1) stacked figure for angular convention `conv`."""
    meta = CONV_META[conv]

    fig, axes = plt.subplots(
        N_bands,
        1,
        figsize=(11, 2.8 * N_bands),
        sharex=True,
        squeeze=False,
    )

    for ax_i, band in enumerate(bands_to_plot):
        ax = axes[ax_i][0]
        bc = BAND_COLORS.get(band, "gray")

        db = df_ang[df_ang["r:band"] == band].sort_values("r:midpointMjdTai")
        mjd = db["r:midpointMjdTai"].values.astype(float)
        ang = apply_convention(db["r:dipoleAngle"].values, conv)
        dip = db["isDipole"].values.astype(bool)
        dlen = db["r:dipoleLength"].values.astype(float)

        # ── Reference guide-lines ─────────────────────────────────────────
        for hl in meta["hlines"]:
            ax.axhline(hl, color="#cccccc", lw=0.8, ls="--", zorder=1)

        # ── Non-dipole visits (open circles) ─────────────────────────────
        mask_nd = ~dip
        if mask_nd.any():
            ax.scatter(
                mjd[mask_nd],
                ang[mask_nd],
                marker="o",
                s=MARKER_SIZE**2,
                facecolors="none",
                edgecolors=bc,
                linewidths=0.9,
                alpha=MARKER_ALPHA * 0.7,
                label="no-dipole",
                zorder=3,
            )

        # ── Dipole visits (filled circles, size ∝ dipoleLength) ───────────
        if dip.any():
            # Scale marker size by dipoleLength (clipped to [4, 16] px radius)
            dlen_dip = dlen[dip]
            sz_scale = np.clip(dlen_dip, 0.5, 8.0) * 10
            ax.scatter(
                mjd[dip],
                ang[dip],
                marker="o",
                s=sz_scale,
                color=bc,
                edgecolors="k",
                linewidths=0.6,
                alpha=MARKER_ALPHA,
                label="isDipole (size ∝ length)",
                zorder=4,
            )
            # Gold ring for emphasis
            ax.scatter(
                mjd[dip],
                ang[dip],
                marker="o",
                s=sz_scale * 1.8,
                facecolors="none",
                edgecolors="gold",
                linewidths=1.1,
                alpha=0.7,
                zorder=5,
            )

        # ── Stats box ─────────────────────────────────────────────────────
        st = band_angle_stats(db, conv)
        _stats_box(ax, st, conv)

        # ── Circular mean line ────────────────────────────────────────────
        ax.axhline(
            st["circ_mean"],
            color=bc,
            lw=1.2,
            ls="-",
            alpha=0.55,
            zorder=2,
            label=f"circ mean = {st['circ_mean']:+.1f}°",
        )

        ax.set_ylabel(f"{meta['label']}\nband {band}", fontsize=8)
        ax.set_ylim(meta["ylim"])
        ax.set_yticks(meta["yticks"])
        ax.tick_params(labelsize=9)
        ax.grid(True, alpha=0.20)
        ax.legend(fontsize=8, ncol=1, bbox_to_anchor=(1.01, 0.5), loc="center left", frameon=False)
        ax.set_title(f"band {band}", fontsize=9, color=bc, fontweight="bold", loc="right")

    axes[-1][0].set_xlabel(f"MJD TAI", fontsize=9)
    # Secondary x-axis: Δt in days
    ax2 = axes[-1][0].twiny()
    ax2.set_xlim(np.array(axes[-1][0].get_xlim()) - T0_MJD)
    ax2.set_xlabel(f"Δt [days from MJD {T0_MJD:.2f} ({T0_DATE})]", fontsize=8)
    ax2.tick_params(labelsize=8)

    dip_str = "isDipole only" if DIPOLE_ONLY else "all visits with finite dipoleAngle"
    fig.suptitle(
        f"Dipole angle vs MJD — {meta['label']}\n"
        f"diaObjectId={DIAOBJECT_ID}  |  {dip_str}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}  "
        f"first={DATE_FIRST}  last={DATE_LAST}",
        fontsize=10,
        y=1.01,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"dipole_angle_stacked_{conv}_obj{DIAOBJECT_ID}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=120)
        print(f"→ saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)


# ── Run for all three conventions ────────────────────────────────────────────
for conv in ("0_360", "-180_180", "mod180"):
    print(f"\n── Stacked figure — convention: {conv} ──")
    plot_stacked(conv)

## 9. Figure B — All bands overlaid in a single subplot (one convention at a time)

Useful for comparing the dipole-angle distribution across bands at a glance.
The primary convention selected by `ANGLE_CONV_PRIMARY` is highlighted first;
the other two conventions follow.

In [ ]:
def plot_overlay(conv):
    """Produce the single-subplot overlay figure for angular convention `conv`."""
    meta = CONV_META[conv]

    fig, ax = plt.subplots(figsize=(12, 5))

    for band in bands_to_plot:
        bc = BAND_COLORS.get(band, "gray")
        db = df_ang[df_ang["r:band"] == band].sort_values("r:midpointMjdTai")
        mjd = db["r:midpointMjdTai"].values.astype(float)
        ang = apply_convention(db["r:dipoleAngle"].values, conv)
        dip = db["isDipole"].values.astype(bool)
        st = band_angle_stats(db, conv)

        # Non-dipole: open circles with band colour
        if (~dip).any():
            ax.scatter(
                mjd[~dip],
                ang[~dip],
                marker="o",
                s=(MARKER_SIZE - 1) ** 2,
                facecolors="none",
                edgecolors=bc,
                linewidths=0.8,
                alpha=0.55,
                zorder=3,
            )

        # Dipole: filled circles
        if dip.any():
            ax.scatter(
                mjd[dip],
                ang[dip],
                marker="o",
                s=MARKER_SIZE**2,
                color=bc,
                edgecolors="k",
                linewidths=0.5,
                alpha=MARKER_ALPHA,
                zorder=4,
            )

        # Circular-mean line per band
        ax.axhline(st["circ_mean"], color=bc, lw=1.0, ls="-", alpha=0.45, zorder=2)

        # Invisible proxy for the legend
        ax.scatter(
            [],
            [],
            marker="o",
            color=bc,
            label=f"{band}  N={st['N']}  μ_c={st['circ_mean']:+.0f}°  σ_c={st['circ_std']:.0f}°  R={st['R']:.2f}",
        )

    # Reference guide-lines
    for hl in meta["hlines"]:
        ax.axhline(hl, color="#cccccc", lw=0.7, ls="--", zorder=1)

    ax.set_xlabel("MJD TAI", fontsize=10)
    ax.set_ylabel(meta["label"], fontsize=10)
    ax.set_ylim(meta["ylim"])
    ax.set_yticks(meta["yticks"])
    ax.tick_params(labelsize=10)
    ax.grid(True, alpha=0.20)
    ax.legend(fontsize=9, ncol=1, bbox_to_anchor=(1.01, 0.5), loc="center left", frameon=True)

    # Secondary x-axis in Δt days
    ax2 = ax.twiny()
    ax2.set_xlim(np.array(ax.get_xlim()) - T0_MJD)
    ax2.set_xlabel(f"Δt [days from MJD {T0_MJD:.2f} ({T0_DATE})]", fontsize=8)
    ax2.tick_params(labelsize=8)

    dip_str = "isDipole only" if DIPOLE_ONLY else "all finite dipoleAngle"
    fig.suptitle(
        f"Dipole angle vs MJD — all bands overlaid — {meta['label']}\n"
        f"diaObjectId={DIAOBJECT_ID}  |  {dip_str}  |  "
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}",
        fontsize=10,
        y=1.04,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"dipole_angle_overlay_{conv}_obj{DIAOBJECT_ID}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=120)
        print(f"→ saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)


# ── Run for all three conventions ────────────────────────────────────────────
for conv in ("0_360", "-180_180", "mod180"):
    print(f"\n── Overlay figure — convention: {conv} ──")
    plot_overlay(conv)

## 10. Bonus — Angle histogram per band (polar + Cartesian)

Two complementary views of the angle distribution:
- **Cartesian histogram** (easy to read frequencies).
- **Polar histogram / rose diagram** (intuitive for orientation data).

Produced for the primary convention and for the physically correct mod-180° case.

In [ ]:
N_BINS_HIST = 18  # bins for the histograms (20° per bin for mod180; 20° for full circle)


def plot_histograms(conv):
    """Cartesian + polar rose histograms for each band."""
    meta = CONV_META[conv]
    y0, y1 = meta["ylim"]
    ang_range = (y0 + 5, y1 - 5)  # trim the 5° padding

    # Each band: one column = (cartesian, polar)
    fig, axes = plt.subplots(
        2,
        N_bands,
        figsize=(3.5 * N_bands, 7),
        subplot_kw={"projection": None},
    )
    # Replace bottom row with polar axes
    for col in range(N_bands):
        axes[1, col].remove()
        axes[1, col] = fig.add_subplot(2, N_bands, N_bands + col + 1, projection="polar")

    for col, band in enumerate(bands_to_plot):
        bc = BAND_COLORS.get(band, "gray")
        db = df_ang[df_ang["r:band"] == band]
        ang = apply_convention(db["r:dipoleAngle"].values, conv)
        ang_valid = ang[np.isfinite(ang)]
        st = band_angle_stats(db, conv)

        # ── Cartesian histogram ──────────────────────────────────────────
        ax_c = axes[0, col]
        bins = np.linspace(ang_range[0], ang_range[1], N_BINS_HIST + 1)
        ax_c.hist(ang_valid, bins=bins, color=bc, edgecolor="white", linewidth=0.5, alpha=0.8)
        ax_c.axvline(
            st["circ_mean"], color="k", lw=1.5, ls="--", label=f"circ mean = {st['circ_mean']:+.1f}°"
        )
        ax_c.set_title(f"band {band}", fontsize=9, color=bc, fontweight="bold")
        ax_c.set_xlabel(meta["label"][:20] + "…", fontsize=7)
        ax_c.set_ylabel("count", fontsize=8)
        ax_c.legend(fontsize=7, loc="upper right")
        ax_c.grid(True, alpha=0.25)

        # ── Polar rose diagram ───────────────────────────────────────────
        ax_p = axes[1, col]
        # Convert to radians for polar plot; adjust range to [0, 2π] or [0, π]
        ang_rad = np.deg2rad(ang_valid)
        bins_rad = np.linspace(np.deg2rad(ang_range[0]), np.deg2rad(ang_range[1]), N_BINS_HIST + 1)
        counts, bin_edges = np.histogram(ang_rad, bins=bins_rad)
        bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        width = bin_edges[1] - bin_edges[0]
        ax_p.bar(bin_centres, counts, width=width, color=bc, edgecolor="white", linewidth=0.4, alpha=0.75)
        # Mean resultant vector
        ax_p.annotate(
            "",
            xy=(np.deg2rad(st["circ_mean"]), st["R"] * counts.max()),
            xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="k", lw=1.5),
        )
        ax_p.set_title(f"R={st['R']:.2f}", fontsize=8, pad=4)
        ax_p.tick_params(labelsize=7)

    fig.suptitle(
        f"Dipole angle distribution — {meta['label']}\n"
        f"diaObjectId={DIAOBJECT_ID}   Gaia G={G_str}   status={gaia_status}",
        fontsize=12,
        y=1.01,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"dipole_angle_hist_{conv}_obj{DIAOBJECT_ID}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=120)
        print(f"→ saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)


for conv in ("-180_180", "mod180"):
    print(f"\n── Histogram — convention: {conv} ──")
    plot_histograms(conv)

## 11. Summary table — dipole angle statistics per band

All three conventions shown together.

In [ ]:
from IPython.display import display as ipy_display

rows = []
for band in bands_to_plot:
    db = df_ang[df_ang["r:band"] == band]
    for conv in ("0_360", "-180_180", "mod180"):
        st = band_angle_stats(db, conv)
        rows.append(
            {
                "band": band,
                "convention": conv,
                "N": st["N"],
                "N_dip": st["N_dip"],
                "circ_mean [°]": round(st["circ_mean"], 2),
                "circ_std [°]": round(st["circ_std"], 2),
                "R (clustering)": round(st["R"], 3),
                "arith_mean [°]": round(st["mean"], 2),
                "arith_std [°]": round(st["std"], 2),
            }
        )

df_stats = pd.DataFrame(rows)
print(f"diaObjectId={DIAOBJECT_ID}  |  Gaia G={G_str}  |  status={gaia_status}  |  group={fink_group}")
ipy_display(df_stats)

## 12. Interpretation notes

### What R tells you

- **R ≈ 1** : all angles nearly identical → dipole consistently points the same direction
  → probable **PSF anisotropy or astrometric offset** fixed in detector coordinates.
- **R ≈ 0** : angles uniformly distributed → dipoles are **random** → perhaps noise spikes
  or different physical causes each visit.
- **R intermediate** : partial clustering → could be a mix, or a slowly rotating systematic.

### Which convention to prefer?

| Use-case | Recommended convention |
|---|---|
| Plotting raw catalogue values | `-180_180` (most natural for scatter around a mean) |
| Checking for 0°/360° wrap artefacts | `0_360` |
| Physical analysis of dipole orientation | `mod180` (the dipole is undirected) |
| Circular statistics | always work in radians; convert back to display units |

> **Tip:** If you want to test whether the angle is **constant across visits**,
> compute the circular standard deviation `σ_c` in the `mod180` convention
> and compare it to the noise floor expected from the SNR of the dipole fit.
